# 🔬 Experimento de Granularidad Extrema: Enriquecimiento por Sección Censal

## 🎯 Objetivo del Cuaderno
Este *notebook* es una variante experimental del proceso de enriquecimiento diseñada para testear la "Maldición de la Dimensionalidad" y la varianza espacial. 

A diferencia del *Baseline* (que utiliza la Renta Media por Distrito), este script realiza un cruce espacial (*Spatial Join*) utilizando archivos *Shapefile* (`.shp`) para mapear cada inmueble a su **Sección Censal** específica (nivel de calle/manzana). El objetivo es generar un *dataset* de alta granularidad para comprobar en la fase de modelado si este exceso de detalle microeconómico mejora la precisión predictiva o si, por el contrario, induce a un sobreajuste del algoritmo.

In [2]:
import numpy as np
import pandas as pd
import matplotlib as plt

In [3]:
df = pd.read_parquet("../DATOS/imagenes_procesadas_total.parquet", engine='fastparquet')

In [4]:
import geopandas as gpd
from shapely.geometry import Point
import numpy as np

# Crear GeoDataFrame desde df
df = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.Longitud, df.Latitud), crs="EPSG:4326")

# Cargar shapefiles
ruido_carreteras = gpd.read_file("../DATOS/mer-carreteras_lden/CDERUI02_Lden_RepGeo_LocalId_C.shp")
ruido_ferroviario = gpd.read_file("../DATOS/mer-lineas_ferroviarias_lden/CDERUI02_Lden_RepGeo_LocalId_F.shp")
ruido_aglomeraciones = gpd.read_file("../DATOS/mer-aglomeraciones_lden/CDERUI02_Lden_RepGeo_LocalId_Ag.shp")
ruido_aeropuertos = gpd.read_file("../DATOS/mer-aeropuertos_lden/CDERUI02_Lden_RepGeo_LocalId_A.shp")

# Reproyectar a sistema métrico (UTM zona 30N)
proj_crs = "EPSG:25830"
df = df.to_crs(proj_crs)
ruido_carreteras = ruido_carreteras.to_crs(proj_crs)
ruido_ferroviario = ruido_ferroviario.to_crs(proj_crs)
ruido_aglomeraciones = ruido_aglomeraciones.to_crs(proj_crs)
ruido_aeropuertos = ruido_aeropuertos.to_crs(proj_crs)

# ---- Aplicar umbrales de distancia
umbral_carretera = 1000     # metros
umbral_ferrocarril = 1500   # metros
umbral_aglomeracion = 1500  # metros
umbral_aeropuerto = 3000    # metros

# ---- JOIN NEAREST 1: Carreteras
temp = df.sjoin_nearest(
    ruido_carreteras[['geometry', 'Lden']],
    how='left',
    distance_col='dist_a_carretera',
    max_distance=umbral_carretera
)
df = temp[~temp.index.duplicated(keep='first')].copy()
df.rename(columns={'Lden': 'ruido_carretera'}, inplace=True)
df.drop(columns=['index_right'], inplace=True, errors='ignore')

# ---- JOIN NEAREST 2: Ferroviario
temp = df.sjoin_nearest(
    ruido_ferroviario[['geometry', 'Lden']],
    how='left',
    distance_col='dist_a_ferrocarril',
    max_distance=umbral_ferrocarril
)
df = temp[~temp.index.duplicated(keep='first')].copy()
df.rename(columns={'Lden': 'ruido_ferrocarril'}, inplace=True)
df.drop(columns=['index_right'], inplace=True, errors='ignore')

# ---- JOIN NEAREST 3: Aglomeraciones
temp = df.sjoin_nearest(
    ruido_aglomeraciones[['geometry', 'Lden']],
    how='left',
    distance_col='dist_a_aglomeracion',
    max_distance=umbral_aglomeracion
)
df = temp[~temp.index.duplicated(keep='first')].copy()
df.rename(columns={'Lden': 'ruido_urbano'}, inplace=True)
df.drop(columns=['index_right'], inplace=True, errors='ignore')

# ---- JOIN NEAREST 4: Aeropuertos
temp = df.sjoin_nearest(
    ruido_aeropuertos[['geometry', 'Lden']],
    how='left',
    distance_col='dist_a_aeropuerto',
    max_distance=umbral_aeropuerto
)
df = temp[~temp.index.duplicated(keep='first')].copy()
df.rename(columns={'Lden': 'ruido_aeropuerto'}, inplace=True)
df.drop(columns=['index_right'], inplace=True, errors='ignore')


df.loc[df['dist_a_carretera'] > umbral_carretera, 'ruido_carretera'] = np.nan
df.loc[df['dist_a_ferrocarril'] > umbral_ferrocarril, 'ruido_ferrocarril'] = np.nan
df.loc[df['dist_a_aglomeracion'] > umbral_aglomeracion, 'ruido_urbano'] = np.nan
df.loc[df['dist_a_aeropuerto'] > umbral_aeropuerto, 'ruido_aeropuerto'] = np.nan

# ---- Calcular promedio solo con fuentes válidas
df['ruido_promedio'] = df[['ruido_carretera', 'ruido_ferrocarril', 'ruido_urbano', 'ruido_aeropuerto']].mean(axis=1)

# ---- Reproyectar a WGS84 para exportar/visualizar
df = df.to_crs("EPSG:4326")

In [5]:
# Definir el valor mínimo de ruido ambiental
ruido_minimo = 30

# Reemplazar NaN y 0.0 en cada columna de ruido
for col in ['ruido_carretera', 'ruido_ferrocarril', 'ruido_urbano', 'ruido_aeropuerto']:
    df[col] = df[col].replace(0.0, np.nan)
    df[col].fillna(ruido_minimo)

# Recalcular el promedio
df['ruido_promedio'] = df[['ruido_carretera', 'ruido_ferrocarril', 'ruido_urbano', 'ruido_aeropuerto']].mean(axis=1)
df['ruido_promedio'].fillna(ruido_minimo)

# Asignar un valor mayor que el umbral cuando no hay fuente cercana
df['dist_a_carretera'] = df['dist_a_carretera'].fillna(umbral_carretera + 100)
df['dist_a_ferrocarril'] = df['dist_a_ferrocarril'].fillna(umbral_ferrocarril + 100)
df['dist_a_aglomeracion'] = df['dist_a_aglomeracion'].fillna(umbral_aglomeracion + 100)
df['dist_a_aeropuerto'] = df['dist_a_aeropuerto'].fillna(umbral_aeropuerto + 100)

df.iloc[:,-10:]

,geometry,ruido_carretera,dist_a_carretera,ruido_ferrocarril,dist_a_ferrocarril,ruido_urbano,dist_a_aglomeracion,ruido_aeropuerto,dist_a_aeropuerto,ruido_promedio
0,POINT (-3.36398 40.48195),NaN,1100.000000,NaN,1600.000000,65.0,0.118166,NaN,3100.0,65.0
1,POINT (-3.57718 40.21105),NaN,1100.000000,NaN,1600.000000,NaN,1600.000000,NaN,3100.0,NaN
2,POINT (-3.43807 40.30077),NaN,1100.000000,55.0,1089.776576,NaN,1600.000000,NaN,3100.0,55.0
3,POINT (-3.43807 40.30077),NaN,1100.000000,55.0,1089.776576,NaN,1600.000000,NaN,3100.0,55.0
4,POINT (-3.86492 40.32385),50.0,764.722856,NaN,1600.000000,NaN,1600.000000,NaN,3100.0,50.0
...,...,...,...,...,...,...,...,...,...,...
1010,POINT (-3.36398 40.48195),NaN,1100.000000,NaN,1600.000000,65.0,0.118166,NaN,3100.0,65.0
1011,POINT (-3.36398 40.48195),NaN,1100.000000,NaN,1600.000000,65.0,0.118166,NaN,3100.0,65.0
1012,POINT (-3.36398 40.48195),NaN,1100.000000,NaN,1600.000000,65.0,0.118166,NaN,3100.0,65.0
1013,POINT (-3.81483 40.43465),50.0,148.344079,55.0,688.211151,NaN,1600.000000,NaN,3100.0,52.5


## 2. Nivel Socioeconómico (Renta Media)
💰 Fuente: Instituto Nacional de Estadística (INE)

✅ Datos por secciones censales o distritos:

Dataset: "Renta media por hogar"

In [9]:
df_rentas = pd.read_csv("../DATOS/rentas.csv", sep = ";", encoding = 'latin1')
df_rentas

,Municipios,Distritos,Secciones,Indicadores de renta media y mediana,Periodo,Total
0,"28001 Acebeda, La",NaN,NaN,Renta neta media por hogar,2023,31.213
1,"28001 Acebeda, La","2800101 Acebeda, La distrito 01",NaN,Renta neta media por hogar,2023,31.213
2,"28001 Acebeda, La","2800101 Acebeda, La distrito 01","2800101001 Acebeda, La sección 01001",Renta neta media por hogar,2023,31.213
3,28002 Ajalvir,NaN,NaN,Renta neta media por hogar,2023,45.022
4,28002 Ajalvir,2800201 Ajalvir distrito 01,NaN,Renta neta media por hogar,2023,45.022
...,...,...,...,...,...,...
4961,28903 Tres Cantos,2890301 Tres Cantos distrito 01,2890301023 Tres Cantos sección 01023,Renta neta media por hogar,2023,66.115
4962,28903 Tres Cantos,2890301 Tres Cantos distrito 01,2890301024 Tres Cantos sección 01024,Renta neta media por hogar,2023,51.834
4963,28903 Tres Cantos,2890301 Tres Cantos distrito 01,2890301025 Tres Cantos sección 01025,Renta neta media por hogar,2023,57.758
4964,28903 Tres Cantos,2890301 Tres Cantos distrito 01,2890301026 Tres Cantos sección 01026,Renta neta media por hogar,2023,51.188


In [26]:
import pandas as pd
import geopandas as gpd

print("🗺️ 1. Cargando y filtrando el mapa del INE...")

ruta_shp = "../DATOS/España_Seccionado2023_ETRS89H30/SECC_CE_20230101.shp"
gdf_mapa = gpd.read_file(ruta_shp)

# El INE guarda el código de 10 dígitos de la sección en una columna llamada 'CUSEC' o 'cusec'
col_cusec = 'CUSEC' if 'CUSEC' in gdf_mapa.columns else 'cusec'

# Filtramos SOLO la Comunidad de Madrid (El código CUSEC de Madrid siempre empieza por '28')
gdf_madrid = gdf_mapa[gdf_mapa[col_cusec].str.startswith('28')].copy()

# TRADUCCIÓN DE COORDENADAS: Pasamos de UTM Huso 30 (INE) a GPS normal (Tus pisos)
# EPSG:4326 es el código estándar mundial para Latitud/Longitud
gdf_madrid = gdf_madrid.to_crs("EPSG:4326")


print("📍 2. Convirtiendo los pisos en un mapa de puntos...")
# (df_final) debe tener las columnas 'Latitud' y 'Longitud'
gdf_pisos = gpd.GeoDataFrame(
    df, 
    geometry=gpd.points_from_xy(df['Longitud'], df['Latitud']),
    crs="EPSG:4326"
)


print("🔗 3. ¡Haciendo el cruce espacial (Spatial Join)!")
# Esto asigna a cada piso el polígono (sección) en el que cae físicamente
pisos_con_seccion = gpd.sjoin(gdf_pisos, gdf_madrid[[col_cusec, 'geometry']], how="left", predicate="within")


import pandas as pd
import geopandas as gpd

print("⚙️ 1. Preparando archivo de rentas con 'Localizacion_Clave'...")


# ==============================================================================
# CÓDIGO DE LIMPIEZA
# ==============================================================================
df_rentas["Municipio_Limpio"] = df_rentas["Municipios"].str.extract(r"\d+\s+(.*)")
df_rentas["Municipio_Limpio"] = df_rentas["Municipio_Limpio"].str.upper().str.strip()

df_rentas["Distrito_Codigo"] = df_rentas["Distritos"].str.extract(r"(28079\d{2})")

codigos_distritos = {
    "2807901": "CENTRO", "2807902": "ARGANZUELA", "2807903": "RETIRO",
    "2807904": "SALAMANCA", "2807905": "CHAMARTIN", "2807906": "TETUAN",
    "2807907": "CHAMBERI", "2807908": "FUENCARRAL-EL PARDO", "2807909": "MONCLOA-ARAVACA",
    "2807910": "LATINA", "2807911": "CARABANCHEL", "2807912": "USERA",
    "2807913": "PUENTE DE VALLECAS", "2807914": "MORATALAZ", "2807915": "CIUDAD LINEAL",
    "2807916": "HORTALEZA", "2807917": "VILLAVERDE", "2807918": "VILLA DE VALLECAS",
    "2807919": "VICALVARO", "2807920": "SAN BLAS-CANILLEJAS", "2807921": "BARAJAS"
}

df_rentas["Distrito_Limpio"] = df_rentas["Distrito_Codigo"].map(codigos_distritos)
df_rentas["Localizacion_Clave"] = df_rentas["Distrito_Limpio"].fillna(df_rentas["Municipio_Limpio"])
# ==============================================================================

# Filtramos la Renta Neta Media por Hogar del año más reciente
df_rentas_hogar = df_rentas[
    (df_rentas['Indicadores de renta media y mediana'] == 'Renta neta media por hogar') & 
    (df_rentas['Secciones'].notna())
].copy()
df_rentas_hogar = df_rentas_hogar[df_rentas_hogar['Periodo'] == df_rentas_hogar['Periodo'].max()]

# Extraemos el código CUSEC y el dinero
df_rentas_hogar['CUSEC_limpio'] = df_rentas_hogar['Secciones'].str[:10]
df_rentas_hogar['Renta_Neta_Seccion'] = pd.to_numeric(
    df_rentas_hogar['Total'].astype(str).str.replace('.', ''), errors='coerce'
)

print("🔗 2. Uniendo la Renta y la Localización Clave a los inmuebles...")
# Asumiendo que ya tienes "pisos_con_seccion" (el resultado del gpd.sjoin del paso anterior)
# Hacemos el merge trayéndonos AMBAS columnas: la Renta y la Localizacion_Clave
df_final_enriquecido = pisos_con_seccion.merge(
    df_rentas_hogar[['CUSEC_limpio', 'Renta_Neta_Seccion', 'Localizacion_Clave']], 
    left_on='CUSEC', # Cambia 'CUSEC' por 'cusec' si tu shapefile lo tiene en minúscula
    right_on='CUSEC_limpio', 
    how='left'
)

# Limpiamos basurilla espacial
df_final = pd.DataFrame(df_final_enriquecido.drop(columns=['geometry', 'index_right', 'CUSEC', 'cusec', 'CUSEC_limpio'], errors='ignore'))

# ==============================================================================
# AUDITORÍA DEFINITIVA: COMPROBANDO LA GRANULARIDAD
# ==============================================================================
print("\n✅ AUDITORÍA: Variación de rentas DENTRO de un mismo distrito")
print("Si el cruce por sección ha funcionado, verás que la Renta Mínima y Máxima dentro de un mismo barrio son muy distintas.")

# Agrupamos por Localizacion_Clave para ver la diversidad de rentas en cada zona
resumen_rentas = df_final.groupby('Localizacion_Clave').agg(
    Num_Pisos=('Renta_Neta_Seccion', 'count'),
    Renta_Minima=('Renta_Neta_Seccion', 'min'),
    Renta_Media=('Renta_Neta_Seccion', 'mean'),
    Renta_Maxima=('Renta_Neta_Seccion', 'max')
).sort_values('Renta_Media', ascending=False)

# Formateamos bonito con el símbolo del euro
formato_euro = lambda x: f"{x:,.0f} €" if pd.notna(x) else "N/A"
resumen_rentas['Renta_Minima'] = resumen_rentas['Renta_Minima'].apply(formato_euro)
resumen_rentas['Renta_Media'] = resumen_rentas['Renta_Media'].apply(formato_euro)
resumen_rentas['Renta_Maxima'] = resumen_rentas['Renta_Maxima'].apply(formato_euro)

# 1. Le decimos a Pandas que NO corte las filas
pd.set_option('display.max_rows', None)

# 2. Mostramos la tabla completa
display(resumen_rentas)

# 3. Restauramos la configuración normal para no saturar futuras celdas
pd.reset_option('display.max_rows')

🗺️ 1. Cargando y filtrando el mapa del INE...
📍 2. Convirtiendo los pisos en un mapa de puntos...
🔗 3. ¡Haciendo el cruce espacial (Spatial Join)!
⚙️ 1. Preparando archivo de rentas con 'Localizacion_Clave'...
🔗 2. Uniendo la Renta y la Localización Clave a los inmuebles...

✅ AUDITORÍA: Variación de rentas DENTRO de un mismo distrito
Si el cruce por sección ha funcionado, verás que la Renta Mínima y Máxima dentro de un mismo barrio son muy distintas.


,Num_Pisos,Renta_Minima,Renta_Media,Renta_Maxima
Localizacion_Clave,,,,
CIUDAD LINEAL,7,"101,278 €","101,278 €","101,278 €"
FUENCARRAL-EL PARDO,4,"99,463 €","99,463 €","99,463 €"
TRES CANTOS,3,"79,764 €","79,764 €","79,764 €"
VILLANUEVA DEL PARDILLO,1,"70,434 €","70,434 €","70,434 €"
COSLADA,21,"67,745 €","67,745 €","67,745 €"
SALAMANCA,26,"44,744 €","64,754 €","73,524 €"
COBEÑA,1,"62,612 €","62,612 €","62,612 €"
TETUAN,31,"36,614 €","57,197 €","104,774 €"
POZUELO DE ALARCÓN,4,"56,742 €","56,742 €","56,742 €"


In [25]:
df_final

,Enlace,Precio,Dormitorios,Superficie,Num_baños,Descripción,Metro,Renfe,Bus,Farmacia,...,dist_a_carretera,ruido_ferrocarril,dist_a_ferrocarril,ruido_urbano,dist_a_aglomeracion,ruido_aeropuerto,dist_a_aeropuerto,ruido_promedio,Renta_Neta_Seccion,Localizacion_Clave
0,https://www.tecnocasa.es/venta/piso/madrid/alc...,368000.0,4,131.0,2,AGENCIA INMOBILIARIA DE ALCALÁ DE HENARES BARR...,0,1,1,1,...,1100.000000,30.0,1600.000000,65.0,0.118166,30.0,3100.0,38.75,39778.0,ALCALÁ DE HENARES
1,https://www.tecnocasa.es/venta/casa/madrid/san...,331000.0,3,139.0,2,TECNOCASA SAN MARTIN DE LA VEGA- Vende viviend...,0,0,1,1,...,1100.000000,30.0,1600.000000,30.0,1600.000000,30.0,3100.0,30.00,38236.0,SAN MARTÍN DE LA VEGA
2,https://www.tecnocasa.es/venta/piso/madrid/arg...,304000.0,3,126.0,2,Agencia inmobiliaria de ARGANDA DEL REY-zona L...,1,0,1,1,...,1100.000000,55.0,1089.776576,30.0,1600.000000,30.0,3100.0,36.25,33418.0,ARGANDA DEL REY
3,https://www.tecnocasa.es/venta/casa/madrid/arg...,450000.0,4,255.0,2,Agencia inmobiliaria de ARGANDA DEL REY-zona L...,1,0,1,1,...,1100.000000,55.0,1089.776576,30.0,1600.000000,30.0,3100.0,36.25,33418.0,ARGANDA DEL REY
4,https://www.tecnocasa.es/venta/piso/madrid/mos...,175000.0,2,46.0,1,Agencia inmobiliaria de MÓSTOLES zona AVENIDA ...,1,1,1,1,...,764.722856,30.0,1600.000000,30.0,1600.000000,30.0,3100.0,35.00,32139.0,MÓSTOLES
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1010,https://www.tecnocasa.es/venta/piso/madrid/alc...,269900.0,2,136.0,2,"AGENCIA INMOBILIARIA DE ALCALÁ DE HENARES, OFI...",0,1,1,1,...,1100.000000,30.0,1600.000000,65.0,0.118166,30.0,3100.0,38.75,39778.0,ALCALÁ DE HENARES
1011,https://www.tecnocasa.es/venta/casa/madrid/alc...,395000.0,6,237.0,3,"AGENCIA INMOBILIARIA DE ALCALÁ DE HENARES, OFI...",0,1,1,1,...,1100.000000,30.0,1600.000000,65.0,0.118166,30.0,3100.0,38.75,39778.0,ALCALÁ DE HENARES
1012,https://www.tecnocasa.es/venta/casa/madrid/alc...,379900.0,4,176.0,1,"AGENCIA INMOBILIARIA DE ALCALÁ DE HENARES, OFI...",0,1,1,1,...,1100.000000,30.0,1600.000000,65.0,0.118166,30.0,3100.0,38.75,39778.0,ALCALÁ DE HENARES
1013,https://www.tecnocasa.es/venta/piso/madrid/poz...,520000.0,2,103.0,2,Agencia inmobiliaria de Pozuelo de Alarcón Est...,0,1,1,1,...,148.344079,55.0,688.211151,30.0,1600.000000,30.0,3100.0,41.25,56742.0,POZUELO DE ALARCÓN


In [24]:
df_final.columns

Index(['Enlace', 'Precio', 'Dormitorios', 'Superficie', 'Num_baños',
       'Descripción', 'Metro', 'Renfe', 'Bus', 'Farmacia', 'Bar',
       'Restaurante', 'Hospital', 'Supermercado', 'Tienda', 'Colegio',
       'Imagenes', 'Vistas', 'Estado_conservacion', 'Garaje', 'Trastero',
       'Piscina', 'Terraza_balcon', 'Seguridad_portero', 'Etiqueta',
       'Amueblado', 'Calle_tranquila', 'Imagenes_locales',
       'Aire acondicionado', 'Tipo_de_inmueble', 'Año_de_construccion',
       'Calefaccion', 'Jardin', 'Ascensor', 'Tipo_vivienda', 'Planta_ordinal',
       'distrito/ciudad', 'Localización_normalizada', 'Latitud', 'Longitud',
       'announcement_id', 'Precio_m2', 'Longitud_desc', 'Riqueza_desc',
       'Polaridad_desc', 'Subjetividad_desc', 'Adjetivos_positivos',
       'Urgencia_desc', 'tipo_suelo', 'material_pared', 'tipo_ventana',
       'altura_techo', 'elementos_estructurales_visibles', 'nivel_lujo',
       'nivel_modernidad', 'estilo_principal', 'calidad_iluminacion',
       '

In [28]:
#df_final.to_parquet("../DATOS/tecnocasa_RentaNetaSeccion.parquet", index=False)